# World Models & Counterfactual Planning

**Level:** Advanced · **Time:** 90 min

In this notebook, we move away from reactive agents that "guess and execute," and build agents that simulate the consequences of their actions in a Digital Twin before touching reality.

We will cover 4 distinct patterns:
1. **The Model-Free Anti-Pattern:** Guessing and failing in production.
2. **The Digital Twin:** Testing actions in a safe state-machine sandbox.
3. **Counterfactual Planning:** Branching into divergent futures (Rollback vs Wait) and scoring them.
4. **The Sim-to-Real Gap:** When the simulation lies about latency.

---
## Pattern 1: The Model-Free Anti-Pattern

A reactive agent guesses that a `drop_table` command is safe. It executes it immediately, causing an outage.

In [1]:
def execute_in_prod(action):
    print(f"  [Prod] Executing: {action}")
    if action == "DROP TABLE users":
        return "🔥 FATAL ERROR: Foreign Key Constraint Violation. Prod is down."
    return "OK"

def reactive_agent():
    print("[Agent] I think dropping the table is a good idea.")
    result = execute_in_prod("DROP TABLE users")
    print(f"[Agent] Result: {result}")

reactive_agent()


[Agent] I think dropping the table is a good idea.
  [Prod] Executing: DROP TABLE users
[Agent] Result: 🔥 FATAL ERROR: Foreign Key Constraint Violation. Prod is down.


---
## Pattern 2: The Digital Twin

The agent spins up an isolated replica of the database, runs the command, and observes the simulated crash safely.

In [2]:
def execute_in_twin(action):
    print(f"  [Twin] Simulating: {action}")
    if action == "DROP TABLE users":
        return "Simulated Error: Foreign Key Constraint."
    return "Simulated Success."

def digital_twin_agent():
    print("[Agent] I want to drop the table. Let me test it in the Twin first.")
    sim_result = execute_in_twin("DROP TABLE users")
    
    if "Error" in sim_result:
        print(f"🚨 [Agent] The simulation failed ({sim_result}). I will NOT execute this in Prod.")
    else:
        print("✅ [Agent] Simulation passed. Safe to execute in Prod.")

digital_twin_agent()


[Agent] I want to drop the table. Let me test it in the Twin first.
  [Twin] Simulating: DROP TABLE users
🚨 [Agent] The simulation failed (Simulated Error: Foreign Key Constraint.). I will NOT execute this in Prod.


---
## Pattern 3: Counterfactual Planning (What If?)

The agent spins up parallel futures, simulating different strategies to find the highest expected utility.

In [3]:
def simulate_future(action):
    if action == "rollback":
        return {"utility": 85, "downtime_mins": 5}
    elif action == "wait":
        return {"utility": 40, "downtime_mins": 15}
    elif action == "hotfix":
        return {"utility": 0, "downtime_mins": 60, "notes": "Compilation failed"}

def counterfactual_agent():
    options = ["rollback", "wait", "hotfix"]
    best_option = None
    max_utility = -1
    
    for option in options:
        print(f"[Agent] Simulating future for: {option}")
        result = simulate_future(option)
        print(f"  -> Predicted Utility: {result['utility']}")
        
        if result['utility'] > max_utility:
            max_utility = result['utility']
            best_option = option
            
    print(f"\n🎯 [Agent] Best option is '{best_option}' with utility {max_utility}. Executing in Prod.")

counterfactual_agent()


[Agent] Simulating future for: rollback
  -> Predicted Utility: 85
[Agent] Simulating future for: wait
  -> Predicted Utility: 40
[Agent] Simulating future for: hotfix
  -> Predicted Utility: 0

🎯 [Agent] Best option is 'rollback' with utility 85. Executing in Prod.


---
## Pattern 4: The Sim-to-Real Gap

If the Digital Twin makes faulty assumptions (e.g., hardcoded latency), the simulation will succeed but reality will fail.

In [4]:
class FlawedTwin:
    def simulate_api_call(self):
        # The twin assumes latency is always 10ms
        assumed_latency = 10
        if assumed_latency < 100:
            return "Simulated Success"

def real_production_api():
    # In reality, the network is congested
    actual_latency = 5000
    if actual_latency > 100:
        return "🔥 FATAL: Timeout Error"

def vulnerable_agent(twin):
    print("[Agent] Testing API call in Twin...")
    sim_result = twin.simulate_api_call()
    
    if "Success" in sim_result:
        print("[Agent] Twin predicts success. Executing in Prod...")
        prod_result = real_production_api()
        print(f"🚨 [Result] {prod_result}. We fell into the Sim-to-Real Gap!")

vulnerable_agent(FlawedTwin())


[Agent] Testing API call in Twin...
[Agent] Twin predicts success. Executing in Prod...
🚨 [Result] 🔥 FATAL: Timeout Error. We fell into the Sim-to-Real Gap!
